# Создание модели классификации

### Загрузка данных и предварительная обработка

In [59]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import (
    accuracy_score, classification_report,
    mean_absolute_error, mean_squared_error, r2_score
)

df = pd.read_csv('patient_segmentation_dataset.csv')
kmeans_model = joblib.load('best_model.kpl')

features_for_clustering = ['Age', 'Annual_Visits', 'Num_Chronic_Conditions']
X_for_clust = df[features_for_clustering]

df['Cluster'] = kmeans_model.predict(X_for_clust.values)

X = df.drop(columns=['Preventive_Care_Flag', 'Patient_ID', 'Full_Name'], errors='ignore')
X = pd.get_dummies(X, drop_first=True)
y = df['Preventive_Care_Flag']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_scaled = scaler.fit_transform(X_for_clust)

D:\Program\Anaconda\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator KMeans from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


### Обучение моделей и оценка качества

In [55]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression, RidgeClassifier

tr = DecisionTreeClassifier(max_depth=4, random_state=42)
tr.fit(X_train, y_train)
tr_pred = tr.predict(X_test)
print("--- Decision Tree ---")
print(classification_report(y_test, tr_pred))

ridge = RidgeClassifier()
ridge.fit(X_train_scaled, y_train)
ridge_pred = ridge.predict(X_test_scaled)
print("--- Ridge Classifier ---")
print(classification_report(y_test, ridge_pred))

--- Decision Tree ---
              precision    recall  f1-score   support

           0       0.54      0.83      0.65       218
           1       0.41      0.14      0.21       182

    accuracy                           0.52       400
   macro avg       0.47      0.48      0.43       400
weighted avg       0.48      0.52      0.45       400

--- Ridge Classifier ---
              precision    recall  f1-score   support

           0       0.55      0.61      0.57       218
           1       0.46      0.40      0.42       182

    accuracy                           0.51       400
   macro avg       0.50      0.50      0.50       400
weighted avg       0.50      0.51      0.51       400



In [51]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
print("Random Forest")
print(classification_report(y_test, rf_pred))

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
knn_pred = knn.predict(X_test_scaled)
print("--- KNN ---")
print(classification_report(y_test, knn_pred))

--- Random Forest ---
              precision    recall  f1-score   support

           0       0.52      0.59      0.55       218
           1       0.41      0.34      0.37       182

    accuracy                           0.48       400
   macro avg       0.46      0.47      0.46       400
weighted avg       0.47      0.48      0.47       400

--- KNN ---
              precision    recall  f1-score   support

           0       0.56      0.75      0.64       218
           1       0.50      0.30      0.38       182

    accuracy                           0.55       400
   macro avg       0.53      0.52      0.51       400
weighted avg       0.53      0.55      0.52       400



Точность довольна низкая из-за малой зависимости данных

### Вывод

Опираясь на получившиеся результаты, как лучшую модель мы выбираем KNN.

## Регресия

In [52]:
print("\nРегрессия \n" + "="*50)

y_reg = df['Cluster']
X_reg = df[features_for_clustering]

X_tr, X_ts, y_tr, y_ts = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

scaler_r = StandardScaler().fit(X_tr)
X_tr = scaler_r.transform(X_tr)
X_ts = scaler_r.transform(X_ts)

# три модели
models_reg = {
    'LinReg': LinearRegression(),
    'RFreg' : RandomForestRegressor(n_estimators=100, random_state=42),
    'GBreg' : GradientBoostingRegressor(n_estimators=100, random_state=42)
}


print(f"{'Модель':<8}  {'MAE':>6}   {'RMSE':>6}   {'R²':>6}")
print("-" * 38)

for name, model in models_reg.items():
    model.fit(X_tr, y_tr)
    pred = model.predict(X_ts)
    
    mae  = mean_absolute_error(y_ts, pred)
    rmse = mean_squared_error(y_ts, pred)
    r2   = r2_score(y_ts, pred)
    
    print(f"{name:<8}  {mae:6.2f}   {rmse:6.2f}   {r2:6.3f}")


Регрессия 
Модель       MAE     RMSE       R²
--------------------------------------
LinReg      0.16     0.11    0.263
RFreg       0.00     0.00    1.000
GBreg       0.00     0.00    1.000


Линейная регрессия показывает самый лучший результат.

### Классификация

In [53]:
models = {
    'KNN': KNeighborsClassifier(5),
    'RF': RandomForestClassifier(100, random_state=42),
    'DT': DecisionTreeClassifier(max_depth=5, random_state=42)
}

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    acc = accuracy_score(y_test, pred)
    print(f"{name:8}  acc: {acc:.4f}")
    print(classification_report(y_test, pred, digits=3))
    print()


KNN       acc: 0.4525
              precision    recall  f1-score   support

           0      0.498     0.555     0.525       218
           1      0.382     0.330     0.354       182

    accuracy                          0.453       400
   macro avg      0.440     0.442     0.439       400
weighted avg      0.445     0.453     0.447       400


RF        acc: 0.4775
              precision    recall  f1-score   support

           0      0.518     0.592     0.552       218
           1      0.411     0.341     0.372       182

    accuracy                          0.477       400
   macro avg      0.464     0.466     0.462       400
weighted avg      0.469     0.477     0.471       400


DT        acc: 0.5275
              precision    recall  f1-score   support

           0      0.541     0.876     0.669       218
           1      0.426     0.110     0.175       182

    accuracy                          0.527       400
   macro avg      0.483     0.493     0.422       400
weight

Дерево принятий решений показывает самый точный результат.

## Сохранение финальной системы

In [62]:
best_clf = KNeighborsClassifier(n_neighbors=5, weights='uniform', metric='minkowski', p=2)
best_clf.fit(X_scaled, y)

,n_neighbors,5
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


In [63]:
joblib.dump(best_clf, 'final_classification_system.pkl')

['final_classification_system.pkl']